In [15]:
import pandas as pd

# Load the dataset
df = pd.read_csv('fake_or_real_news.csv')

# Display the first 5 rows
print("First 5 rows of the DataFrame:")
print(df.head())

# Print the shape of the DataFrame
print("\nShape of the DataFrame:", df.shape)

# Check for missing values
print("\nMissing values in each column:")
print(df.isnull().sum())

First 5 rows of the DataFrame:
   Unnamed: 0                                              title  \
0        8476                       You Can Smell Hillary’s Fear   
1       10294  Watch The Exact Moment Paul Ryan Committed Pol...   
2        3608        Kerry to go to Paris in gesture of sympathy   
3       10142  Bernie supporters on Twitter erupt in anger ag...   
4         875   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  

Shape of the DataFrame: (6335, 4)

Missing values in each column:
Unnamed: 0    0
title         0
text          0
label         0
dtype: int64


In [16]:
import pandas as pd

# Load the dataset with the 'python' engine to handle potential parsing issues
df = pd.read_csv('fake_or_real_news.csv', engine='python', encoding='utf-8')

# Display the first 5 rows
print("First 5 rows of the DataFrame:")
print(df.head())

# Print the shape of the DataFrame
print("\nShape of the DataFrame:", df.shape)

# Check for missing values
print("\nMissing values in each column:")
print(df.isnull().sum())

First 5 rows of the DataFrame:
   Unnamed: 0                                              title  \
0        8476                       You Can Smell Hillary’s Fear   
1       10294  Watch The Exact Moment Paul Ryan Committed Pol...   
2        3608        Kerry to go to Paris in gesture of sympathy   
3       10142  Bernie supporters on Twitter erupt in anger ag...   
4         875   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  

Shape of the DataFrame: (6335, 4)

Missing values in each column:
Unnamed: 0    0
title         0
text          0
label         0
dtype: int64


**Reasoning**:
The previous attempt to load the CSV using `engine='python'` still resulted in a `ParserError: unexpected end of data`. This indicates malformed rows that even the Python engine can't fully interpret. To proceed with loading the dataset and skipping these problematic lines, I will add the `on_bad_lines='skip'` parameter to `pd.read_csv()`.



In [9]:
import pandas as pd
df = pd.read_csv('fake_or_real_news.csv', engine='python', encoding='utf-8', on_bad_lines='skip')
print("First 5 rows of the DataFrame:")
print(df.head())
print("\nShape of the DataFrame:", df.shape)
print("\nMissing values in each column:")
print(df.isnull().sum())

First 5 rows of the DataFrame:
   Unnamed: 0                                              title  \
0        8476                       You Can Smell Hillary’s Fear   
1       10294  Watch The Exact Moment Paul Ryan Committed Pol...   
2        3608        Kerry to go to Paris in gesture of sympathy   
3       10142  Bernie supporters on Twitter erupt in anger ag...   
4         875   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  

Shape of the DataFrame: (6335, 4)

Missing values in each column:
Unnamed: 0    0
title         0
text          0
label         0
dtype: int64


In [10]:
from sklearn.model_selection import train_test_split
X = df['text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)
print("\ny_train value counts:\n", y_train.value_counts(normalize=True))
print("\ny_test value counts:\n", y_test.value_counts(normalize=True))

Shape of X_train: (5068,)
Shape of X_test: (1267,)
Shape of y_train: (5068,)
Shape of y_test: (1267,)

y_train value counts:
 label
REAL    0.500592
FAKE    0.499408
Name: proportion, dtype: float64

y_test value counts:
 label
REAL    0.500395
FAKE    0.499605
Name: proportion, dtype: float64


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)
print("Shape of TF-IDF training data:", tfidf_train.shape)
print("Shape of TF-IDF test data:", tfidf_test.shape)

Shape of TF-IDF training data: (5068, 61595)
Shape of TF-IDF test data: (1267, 61595)


In [12]:
from sklearn.linear_model import PassiveAggressiveClassifier
pac = PassiveAggressiveClassifier(random_state=42)
pac.fit(tfidf_train, y_train)
print("PassiveAggressiveClassifier model trained successfully.")

PassiveAggressiveClassifier model trained successfully.


In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix
y_pred = pac.predict(tfidf_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.4f}")
cm = confusion_matrix(y_test, y_pred, labels=['FAKE', 'REAL'])
print("\nConfusion Matrix:")
print(cm)

Accuracy Score: 0.9408

Confusion Matrix:
[[598  35]
 [ 40 594]]


In [14]:
def predict_news_authenticity(news_text):
    news_tfidf = tfidf_vectorizer.transform([news_text])
    prediction = pac.predict(news_tfidf)
    return prediction[0]
print("\n--- Testing with Custom News Texts ---")
fake_news_example = "BREAKING: Hillary Clinton caught selling state secrets to foreign adversaries for personal gain, sources say. Mainstream media silent on this shocking development."
print(f"News: '{fake_news_example[:70]}...'\nPrediction: {predict_news_authenticity(fake_news_example)}")
real_news_example = "The Supreme Court is expected to hear arguments on a landmark case concerning environmental regulations next month. Legal experts anticipate a divided court."
print(f"\nNews: '{real_news_example[:70]}...'\nPrediction: {predict_news_authenticity(real_news_example)}")


--- Testing with Custom News Texts ---
News: 'BREAKING: Hillary Clinton caught selling state secrets to foreign adve...'
Prediction: FAKE

News: 'The Supreme Court is expected to hear arguments on a landmark case con...'
Prediction: FAKE
